In [ ]:
from utils import * 

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from torch.quantization import quantize_dynamic
import gc
from typing import List, Tuple, Dict, Any
import math
from tqdm.auto import tqdm

# choice embeddings
def load_embeddings(
    sub_id,
    *,
    neutrals=True,
    context="in-context",  # {"in-context", "no-context"}
    llm="openai",
    on_missing="none",     # {"none","empty","raise"}
):
    """
    Load decision embeddings for a subject.

    Directory selection (based on sub_id AFTER stripping out 'sub-'):
      - If int in [1, 100]:  ../data/other-samples/tavares/narratives/narrative-embeddings
      - If non-int:         ../data/other-samples/online/narratives/narrative-embeddings
      - Otherwise:          ../data/narratives/narrative-embeddings

    Subject ID normalization:
      - Always uses "sub-" prefix
      - If the id is an integer in [1, 9], it is zero-padded to 2 digits (1 -> "01")
      - Otherwise, uses the original digits (18001 -> "18001")

    File naming (matches your existing convention):
      sub-<id>_decisions-in-context_<llm>.npz
      sub-<id>_decisions-no-context_<llm>.npz

    Neutral handling:
      - If neutrals=False, applies `drop_neutral_trials()` to the loaded embedding array.

    Missing-file behavior via `on_missing`:
      - "none"  -> return None
      - "empty" -> return np.empty((0, 0), float)
      - "raise" -> raise FileNotFoundError
    """
    def _strip_prefix(x) -> str:
        s = str(x).strip()
        return s[4:] if s.startswith("sub-") else s

    def _parse_int(s: str):
        s = s.strip()
        return int(s) if s.isdigit() else None

    def _format_sid(x) -> str:
        """
        Returns normalized subject id string WITH 'sub-' prefix,
        applying 2-digit zero pad only for integer ids 1..9.
        """
        raw = _strip_prefix(x)
        iv = _parse_int(raw)
        if iv is not None:
            core = f"{iv:02d}" if 1 <= iv <= 9 else str(iv)
        else:
            core = raw
        return f"sub-{core}"

    # normalize sub_id used in filename
    sid = _format_sid(sub_id)

    # choose base dir (keep your routing logic)
    base_dir_inlab   = "../data/narratives/narrative-embeddings"
    base_dir_online  = "../data/other-samples/online/narratives/narrative-embeddings"
    base_dir_tavares = "../data/other-samples/tavares/narratives/narrative-embeddings"

    raw = _strip_prefix(sub_id)
    iv = _parse_int(raw)
    if iv is not None and 1 <= iv <= 100:
        base_dir = base_dir_tavares
    elif iv is None:
        base_dir = base_dir_online
    else:
        base_dir = base_dir_inlab

    # filename
    if context not in {"in-context", "no-context"}:
        raise ValueError("context must be 'in-context' or 'no-context'")
    ctxt_suffix = "decisions-in-context" if context == "in-context" else "decisions-no-context"
    fpath = os.path.join(base_dir, f"{sid}_{ctxt_suffix}_{llm}.npz")
    if not os.path.exists(fpath):
        if on_missing == "raise":
            raise FileNotFoundError(f"Embeddings not found: {fpath}")
        if on_missing == "empty":
            return np.empty((0, 0), dtype=float)
        return None  # on_missing == "none"

    # load and drop neutrals if requested
    arr = np.load(fpath)["embedding"].astype(float)
    if not neutrals:
        arr = drop_neutral_trials(arr)

    return arr

# full narrative text
def load_narrative(
    sub_id,
    *,
    on_missing="none",  # {"none","empty","raise"}
):
    """
    Load subject narrative data from:
        ../data/narratives/narratives/<sub_id>.xlsx

    Subject ID normalization:
      - Always returns/uses a "sub-" prefixed ID in the filename.
      - If the id is an integer in [1, 9], it is zero-padded to 2 digits (1 -> "sub-01").
      - If the id is an integer >= 10, uses the digits as-is (18001 -> "sub-18001").
      - If the id is non-integer (e.g., "online_abc"), uses it as-is (-> "sub-online_abc")
        unless it already starts with "sub-".

    Missing-file behavior via `on_missing`:
      - "none"  -> return None
      - "empty" -> return empty DataFrame
      - "raise" -> raise FileNotFoundError
    """
    def _strip_prefix(x) -> str:
        s = str(x).strip()
        return s[4:] if s.startswith("sub-") else s

    def _parse_int(s: str):
        s = s.strip()
        return int(s) if s.isdigit() else None

    def _format_sid(x) -> str:
        raw = _strip_prefix(x)
        iv = _parse_int(raw)
        if iv is not None:
            core = f"{iv:02d}" if 1 <= iv <= 9 else str(iv)
        else:
            core = raw
        return f"sub-{core}"

    sid = _format_sid(sub_id)

    base_dir = "../data/narratives/narratives"
    fpath = os.path.join(base_dir, f"{sid}.xlsx")

    if not os.path.exists(fpath):
        if on_missing == "raise":
            raise FileNotFoundError(f"Narrative file not found: {fpath}")
        if on_missing == "empty":
            return pd.DataFrame()
        return None  # on_missing == "none"

    return pd.read_excel(fpath)


<h1 align='center'> Compute internal features </h1>

- Entropy of attention
- Negative log likelihood
- etc...

In [8]:
def get_inputs(sub_id):
    # organize the inputs
    inputs = list(load_narrative(sub_id)['text'].values)
    return inputs

def load_model(model):

    # get the full identifier
    print(f'Loading {model}')
    if 'qwen' in model.lower():
        model_name = f'Qwen/{model}'
    elif 'llama' in model.lower():
        model_name = f'meta-llama/{model}'
    elif 'gemma' in model.lower():
        model_name = f'google/{model}'
    elif 'phi' in model.lower():
        model_name = f'microsoft/{model}'

    # load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None: # ensure there is a pad token; if not, default it to the EOS token
        tokenizer.pad_token = tokenizer.eos_token

    # load model
    llm = AutoModelForCausalLM.from_pretrained(model_name,
                                                low_cpu_mem_usage=True, # streams weights off CPU to save peak RAM
                                                attn_implementation="eager", # forces the safe, PyTorch-native attention path
                                                return_dict_in_generate=True, # get structured outputs from .generate()
                                                output_attentions=True) # capture attention weights for analysis

    # quantization for less memory and speed up on cpu
    torch.backends.quantized.engine = 'qnnpack'
    llm = quantize_dynamic(llm,
                            {torch.nn.Linear}, # only quantize Linear modules
                            dtype=torch.qint8 # convert weights from 32 bit into signed 8-bit integers
                            ).eval()
    
    return tokenizer, llm

# to compute a forward pass in batches
def forward_batch(texts: List[str], tokenizer, llm):
    """
    Return tokenized inputs + outputs needed for analyses.
    Shapes:
      input_ids        : (B, T)
      attention_mask   : (B, T)
      last_tokens      : (B, T, D)
      attentions       : list/tuple length L, each (B, H, T, T)
      logits           : (B, T, V)
    """
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
    with torch.inference_mode():
        out = llm(
            **inputs,
            output_hidden_states=True,
            output_attentions=True,
            return_dict=True,
        )

    # core tensors
    input_ids      = inputs["input_ids"]
    attention_mask = inputs["attention_mask"].float()
    last_tokens    = out.hidden_states[-1]
    attentions     = out.attentions
    logits         = out.logits

    # shape checks
    B, T = input_ids.shape
    assert last_tokens.shape[:2] == (B, T), f"last_tokens {last_tokens.shape} vs {(B,T)}"
    if len(attentions) > 0:
        L = len(attentions)
        H = attentions[0].shape[1]
        for l in range(L):
            A = attentions[l]
            assert A.shape == (B, H, T, T), f"attentions[{l}] shape {A.shape} != {(B,H,T,T)}"

    return input_ids, attention_mask, last_tokens, attentions, logits

# dsifferent statsitcis to compute
def mean_pooled_embedding(last_tokens: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    """
    last_tokens: (B, T, D)
    attention_mask: (B, T) float
    returns emb_mean: (B, D)
    """
    B, T, D = last_tokens.shape
    mask = attention_mask
    emb_mean = (last_tokens * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True).clamp_min(1.0)
    assert emb_mean.shape == (B, D)
    return emb_mean

def compute_nll_mean(logits: torch.Tensor, input_ids: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """
    logits: (B, T, V)
    input_ids: (B, T)
    mask: (B, T) float/bool
    returns nll_mean: (B,)
    """
    logp   = logits[:, :-1, :].log_softmax(-1)                # (B, T-1, V)
    labels = input_ids[:, 1:].unsqueeze(-1)                   # (B, T-1, 1)
    mask_s = (mask[:, 1:] > 0.5)                              # (B, T-1) bool

    nll = -logp.gather(-1, labels).squeeze(-1)                # (B, T-1)
    nll = nll * mask_s.float()
    lengths = mask_s.sum(dim=1).clamp_min(1)
    nll_mean = nll.sum(dim=1) / lengths
    assert nll_mean.shape[0] == logits.shape[0]
    return nll_mean

def last_token_attn_entropy(
    attentions: List[torch.Tensor],    # length L, each (B,H,T,T)
    attention_mask: torch.Tensor,      # (B,T) float/bool
    *,
    topk: int | None = None,
    normalize: bool = True,
    eps: float = 1e-12
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Returns:
      ent_avg : (B,)      mean across (L,H) of last-token entropy per sequence
      ent_lh  : (B,L,H)   last-token entropy per (layer,head)
    """
    if isinstance(attentions, tuple):
        attentions = list(attentions)
    B, T = attention_mask.shape
    L = len(attentions)
    H = attentions[0].shape[1]
    dev = attention_mask.device

    mask_bool = attention_mask.bool() if attention_mask.dtype == torch.bool else (attention_mask > 0.5)
    ent_lh = torch.empty((B, L, H), device=dev)

    for b in range(B):
        T_real = int(mask_bool[b].sum().item())
        if T_real <= 0:
            ent_lh[b].fill_(0)
            continue

        support = min(topk, T_real) if (topk is not None) else T_real
        denom = math.log(max(support, 1))
        denom = denom if normalize and denom > 0 else 1.0
        denom = torch.tensor(denom, device=dev, dtype=torch.float32)

        for ell, A_l in enumerate(attentions):
            row = A_l[b, :, T_real-1, :T_real]  # (H, T_real)
            row_sum = row.sum(-1, keepdim=True)
            is_prob = torch.allclose(row_sum, torch.ones_like(row_sum), rtol=1e-3, atol=1e-3) \
                      and (row.min() >= -1e-6) and (row.max() <= 1+1e-6)
            P = row / row_sum.clamp_min(eps) if is_prob else torch.softmax(row, dim=-1)

            if topk is not None and topk > 0:
                k = min(int(topk), T_real)
                vals, _ = torch.topk(P, k=k, dim=-1)              # (H, k)
                P_eff = vals / vals.sum(-1, keepdim=True).clamp_min(eps)
            else:
                P_eff = P

            H_last = -(P_eff * (P_eff + eps).log()).sum(-1)       # (H,)
            ent_lh[b, ell] = H_last / denom

    ent_avg = ent_lh.mean(dim=(1, 2))                             # (B,)
    assert ent_avg.shape == (B,)
    assert ent_lh.shape == (B, L, H)
    return ent_avg, ent_lh

def last_token_attn_perplexity(ent_lh: torch.Tensor, *, normalized: bool = False, support: int | None = None) -> torch.Tensor:
    """
    ent_lh: (B, L, H) entropies (natural log base).
    If normalized=False: perplexity = exp(H).
    If normalized=True:  report perplexity / support (i.e., fraction of max support used).
      -> you must pass 'support' = T_real or k (for topk).
    Returns (B, L, H)
    """
    if normalized:
        assert support is not None and support >= 1
        return torch.exp(ent_lh) / float(support)
    else:
        return torch.exp(ent_lh)

def last_token_attn_kept_mass(
    attentions: List[torch.Tensor],
    attention_mask: torch.Tensor,
    k: int,
    eps: float = 1e-12
) -> torch.Tensor:
    """
    Fraction of probability mass captured by the top-k keys of the last-token row.
    Returns kept_mass: (B, L, H) in [0,1].
    """
    if isinstance(attentions, tuple):
        attentions = list(attentions)
    B, T = attention_mask.shape
    L = len(attentions)
    H = attentions[0].shape[1]

    mask_bool = attention_mask.bool() if attention_mask.dtype == torch.bool else (attention_mask > 0.5)
    kept = torch.empty((B, L, H), device=attention_mask.device)

    for b in range(B):
        T_real = int(mask_bool[b].sum().item())
        if T_real <= 0:
            kept[b].fill_(0); continue
        kk = min(int(k), T_real)

        for ell, A_l in enumerate(attentions):
            row = A_l[b, :, T_real-1, :T_real]                    # (H, T_real)
            row_sum = row.sum(-1, keepdim=True).clamp_min(eps)
            P = row / row_sum                                     # probs
            vals, _ = torch.topk(P, k=kk, dim=-1)                 # (H, kk)
            kept[b, ell] = vals.sum(-1)                           # (H,)
    assert kept.shape == (B, L, H)
    return kept

def attn_mean_row_entropy_over_tokens(
    attentions: List[torch.Tensor],
    attention_mask: torch.Tensor,
    eps: float = 1e-12,
    normalize: bool = True
) -> torch.Tensor:
    """
    Mean row entropy across all real query tokens (not just last).
    Returns ent_mean: (B, L, H) normalized by log(T_real) if normalize=True.
    """
    if isinstance(attentions, tuple):
        attentions = list(attentions)
    B, T = attention_mask.shape
    L = len(attentions)
    H = attentions[0].shape[1]
    dev = attention_mask.device
    ent_mean = torch.empty((B, L, H), device=dev)

    mask_bool = attention_mask.bool() if attention_mask.dtype == torch.bool else (attention_mask > 0.5)

    for b in range(B):
        T_real = int(mask_bool[b].sum().item())
        if T_real <= 0:
            ent_mean[b].fill_(0); continue

        denom = math.log(T_real) if (normalize and T_real > 1) else 1.0
        denom = torch.tensor(denom, device=dev, dtype=torch.float32)

        for ell, A_l in enumerate(attentions):
            A = A_l[b, :, :T_real, :T_real]                       # (H, T_real, T_real)
            row_sum = A.sum(-1, keepdim=True).clamp_min(eps)
            P = A / row_sum                                       # row-normalize
            H_rows = -(P.clamp_min(eps) * (P.clamp_min(eps)).log()).sum(-1)  # (H, T_real)
            ent_mean[b, ell] = H_rows.mean(-1) / denom            # (H,)
    assert ent_mean.shape == (B, L, H)
    return ent_mean

def last_token_attn_mean_distance_back(
    attentions: List[torch.Tensor],
    attention_mask: torch.Tensor
) -> torch.Tensor:
    """
    Expected key distance for the last token (how far back we look).
    Returns mean_d: (B, L, H), where distance for query i to key j is (i - j).
    """
    if isinstance(attentions, tuple):
        attentions = list(attentions)
    B, T = attention_mask.shape
    L = len(attentions)
    H = attentions[0].shape[1]
    dev = attention_mask.device
    mean_d = torch.empty((B, L, H), device=dev)

    mask_bool = attention_mask.bool() if attention_mask.dtype == torch.bool else (attention_mask > 0.5)

    for b in range(B):
        T_real = int(mask_bool[b].sum().item())
        if T_real <= 0:
            mean_d[b].fill_(0); continue
        d = torch.arange(T_real, device=dev)  # 0..T_real-1

        for ell, A_l in enumerate(attentions):
            row = A_l[b, :, T_real-1, :T_real]                    # (H, T_real)
            P = row / row.sum(-1, keepdim=True).clamp_min(1e-12)  # probs
            # last query index = T_real-1; distance is (T_real-1 - j)
            dist = (T_real - 1) - d.view(1, -1)                   # (1, T_real)
            mean_d[b, ell] = (P * dist).sum(-1)                   # (H,)
    assert mean_d.shape == (B, L, H)
    return mean_d

# putting it all together
def summarize_attention(attentions, attention_mask, *, topk_keep_mass=32, normalize=True):
    ent_avg, ent_lh = last_token_attn_entropy(attentions, attention_mask, topk=None, normalize=normalize)
    ent_mean_rows   = attn_mean_row_entropy_over_tokens(attentions, attention_mask, normalize=normalize)
    kept_mass_topk  = last_token_attn_kept_mass(attentions, attention_mask, k=topk_keep_mass)
    last_mean_dist  = last_token_attn_mean_distance_back(attentions, attention_mask)

    B, T = attention_mask.shape
    L = len(attentions); H = attentions[0].shape[1]

    # shape checks
    assert ent_lh.shape == (B, L, H)
    assert ent_mean_rows.shape == (B, L, H)
    assert kept_mass_topk.shape == (B, L, H)
    assert last_mean_dist.shape == (B, L, H)
    assert ent_avg.shape == (B,)

    # 👇 standardized keys
    return dict(
        ent_lh=ent_lh,                 # (B,L,H)
        ent_avg=ent_avg,               # (B,)
        ent_mean_rows=ent_mean_rows,   # (B,L,H)
        kept_mass_topk=kept_mass_topk, # (B,L,H)
        last_mean_distance=last_mean_dist,  # (B,L,H)
    )

def load_records_from_npz(npz_path: str) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    """
    Load the previously saved summaries and return:
      - records: List[dict] (same schema as you append during batching)
      - meta:    dict with L/H/D sizes (if present), else {}
    """
    arr = np.load(npz_path, allow_pickle=True)
    ids   = arr["observation_context_id"]
    Treal = arr["T_real"]
    nll   = arr["nll_mean"]
    emb   = arr["emb_mean"]
    ent_a = arr["ent_avg"]
    ent_lh= arr["ent_lh"]
    ent_rows = arr["ent_mean_rows"]
    kept_k   = arr["kept_mass_top32"]
    mean_dst = arr["last_mean_distance"]

    N = len(ids)
    records: List[Dict[str, Any]] = []
    for i in range(N):
        records.append({
            "observation_context_id": ids[i],
            "T_real":             int(Treal[i]),
            "nll_mean":           float(nll[i]),
            "emb_mean":           emb[i],           # (D,) float16
            "ent_avg":            float(ent_a[i]),
            "ent_lh":             ent_lh[i],        # (L,H) float16
            "ent_mean_rows":      ent_rows[i],      # (L,H) float16
            "kept_mass_top32":    kept_k[i],        # (L,H) float16
            "last_mean_distance": mean_dst[i],      # (L,H) float16
        })
    meta = {}
    if "meta" in arr:
        try:
            meta = arr["meta"][0].item()
        except Exception:
            meta = {}
    return records, meta

def save_summaries_npz(records: List[Dict[str, Any]], out_path: str) -> None:
    if not records:
        print("save_summaries_npz: nothing to save.")
        return

    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)

    ids   = np.array([r["observation_context_id"] for r in records], dtype=object)
    emb   = np.stack([r["emb_mean"]           for r in records], axis=0)            # (N,D) fp16
    nll   = np.array([r["nll_mean"]           for r in records], dtype=np.float32)  # (N,)
    ent_a = np.array([r["ent_avg"]            for r in records], dtype=np.float32)  # (N,)
    Treal = np.array([r["T_real"]             for r in records], dtype=np.int32)    # (N,)

    ent_lh        = np.stack([r["ent_lh"]        for r in records], axis=0)  # (N,L,H) fp16
    ent_mean_rows = np.stack([r["ent_mean_rows"] for r in records], axis=0)  # (N,L,H) fp16
    kept_mass_top32    = np.stack([r["kept_mass_top32"]    for r in records], axis=0)  # (N,L,H) fp16
    last_mean_distance = np.stack([r["last_mean_distance"] for r in records], axis=0)  # (N,L,H) fp16

    # sanity shapes
    N, D = emb.shape
    _, L, H = ent_lh.shape
    assert ent_mean_rows.shape == (N, L, H)
    assert kept_mass_top32.shape == (N, L, H)
    assert last_mean_distance.shape == (N, L, H)

    meta = dict(N=int(N), D=int(D), L=int(L), H=int(H))

    tmp = out_path + ".tmp.npz"  # must end with .npz so np.savez_compressed doesn't append again
    np.savez_compressed(
        tmp,
        observation_context_id = ids,
        T_real      = Treal,
        nll_mean    = nll,
        emb_mean    = emb,
        ent_avg     = ent_a,
        ent_lh             = ent_lh,
        ent_mean_rows      = ent_mean_rows,
        kept_mass_top32    = kept_mass_top32,
        last_mean_distance = last_mean_distance,
        meta       = np.array([meta], dtype=object),
    )
    os.replace(tmp, out_path)
    print(f"Saved {N} contexts to {out_path}  (L={L}, H={H}, D={D})")

def dedupe_records(records: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    by_id: Dict[Any, Dict[str, Any]] = {}
    for r in records:
        by_id[r["observation_context_id"]] = r  # last write wins
    return list(by_id.values())

def normalize_inputs(raw_inputs: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Accepts entries either with 'observation_context_id' or 'obs_ctxt_id' and returns
    [{'observation_context_id': ..., 'text': ...}, ...]
    """
    out = []
    for d in raw_inputs:
        cid = d.get("observation_context_id", d.get("obs_ctxt_id"))
        assert cid is not None, f"Missing context id in item: {d.keys()}"
        out.append({"observation_context_id": cid, "text": d["text"]})
    return out

def run_batches_and_collect(raw_inputs, tokenizer, llm, batch_size=64):
    inputs = normalize_inputs(raw_inputs)  # ensures observation_context_id + text

    records = []
    for start in ...:
        batch = inputs[start:start+batch_size]
        texts = [d["text"] for d in batch]
        ctxt_ids = [d["observation_context_id"] for d in batch]

        input_ids, mask, last_tokens, atts, logits = forward_batch(texts, tokenizer, llm)

        emb_mean = mean_pooled_embedding(last_tokens, mask)
        nll_mean = compute_nll_mean(logits, input_ids, mask)

        attn_summ = summarize_attention(atts, mask, topk_keep_mass=32, normalize=True)
        ent_lh    = attn_summ["ent_lh"]
        ent_avg   = attn_summ["ent_avg"]
        ent_rows  = attn_summ["ent_mean_rows"]
        kept_topk = attn_summ["kept_mass_topk"]
        mean_dist = attn_summ["last_mean_distance"]

        for b in range(B):
            records.append({
                "observation_context_id": ctxt_ids[b],
                "T_real": int(mask[b].sum().item()),
                "nll_mean": float(nll_mean[b].item()),
                "emb_mean": emb_mean[b].cpu().numpy().astype(np.float16),
                "ent_avg": float(ent_avg[b].item()),
                "ent_lh": ent_lh[b].cpu().numpy().astype(np.float16),
                "ent_mean_rows": ent_rows[b].cpu().numpy().astype(np.float16),
                "kept_mass_top32": kept_topk[b].cpu().numpy().astype(np.float16),
                "last_mean_distance": mean_dist[b].cpu().numpy().astype(np.float16),
            })
    return records
